# Linear mixed model GWAS

> **Important notes for this notebook**
>
> In `GWAS5`, we tested one SNP at a time with linear and logistic
> regression (`plink --linear` and `--logistic`). This required removing
> related individuals, and it treated the combined effect of all other
> SNPs as noise. In this notebook, we use a **linear mixed model (LMM)**
> GWAS, **LDAK-KVIK** ([Hof et al.,
> 2024](https://doi.org/10.1038/s41588-024-01986-w)). It first predicts
> the effect of the rest of the genome, removes it from the phenotype,
> and then tests each SNP. This gives more power, allows related
> individuals, and runs fast on biobank-sized data.
>
> As in `GWAS8`, we use the simulated cohort `Data/SIM.QC` ($N = 5,000$
> individuals, 50,000 SNPs). Mixed models only gain power when they can
> predict part of the phenotype from the genome, and that needs
> thousands of individuals.
>
> ## Learning outcomes
>
> - **Explain** how standard single-SNP regression (`PLINK`) differs
>   from mixed-model methods (`GCTA`, `BOLT-LMM`, `REGENIE`,
>   `LDAK-KVIK`)
> - **Explain** how a Leave-One-Chromosome-Out (`LOCO`) prediction
>   increases power without removing the signal of the tested SNP
> - **Run** a two-step mixed-model GWAS with `LDAK-KVIK` (`--kvik-step1`
>   and `--kvik-step2`) for a quantitative trait
> - **Run** `LDAK-KVIK` on a case-control trait (`--binary YES`) and
>   **explain** why the Saddlepoint Approximation (SPA) matters when
>   cases are rare
> - **Compare** the power and calibration of `PLINK` and `LDAK-KVIK` in
>   `R`
>
> ## How to make this notebook work
>
> - In this notebook, we will use both `bash command line` and `R`
>   programming languages. Remember to **change the kernel** whenever
>   you transition from one language to the other
>   (`Kernel --> Change Kernel`) indicated by the languages’ images. We
>   will first run `Bash` commands.
>
> <img src="Images/bash.png" alt="Bash" width="80"> Choose the Bash
> kernel

> **Some theory**
>
> ### Intuition First: How does a Mixed Model boost GWAS power?
>
> Imagine trying to hear a **solo singer** (one SNP on chromosome 1)
> while a **background choir** sings (the combined effect of all causal
> SNPs on chromosomes 2–22). Standard regression (`plink --linear`)
> treats the choir as noise, which makes the singer hard to hear. A
> mixed-model GWAS such as `LDAK-KVIK` works in two steps:
>
> 1.  **Step 1 — quiet the choir**: Fit all SNPs on chromosomes 2–22
>     together to predict each person’s genetic background,
>     $\hat{g}^{(-1)}$, and remove it from the phenotype. This is called
>     a **Leave-One-Chromosome-Out (`LOCO`)** prediction.
> 2.  **Step 2 — listen to the singer**: Test each SNP on chromosome 1
>     against the “quieter” phenotype. With less background noise, true
>     causal SNPs give larger test statistics.
>
> The same is done for every chromosome. Because the genetic background
> is modelled, close relatives can also stay in the analysis instead of
> being removed as in `GWAS3`.
>
> ### Why move beyond standard single-SNP GWAS (`PLINK`)?
>
> In `GWAS5`, we tested each SNP $j$ on its own:
>
> $$y = X_j \beta_j + Z \gamma + \epsilon, \quad \epsilon \sim \mathcal{N}(0, \sigma_e^2 I)$$
>
> where $y$ is the phenotype of the $N$ individuals, $X_j$ the genotypes
> at SNP $j$, and $Z$ the covariates (e.g., sex and principal
> components). This has three problems in large modern cohorts:
>
> 1.  **Relatives must be removed**: The model assumes independent
>     residuals, which is not true for relatives, so in `GWAS3` we
>     removed related individuals ($\hat{\pi} > 0.125$). In the UK
>     Biobank, about 30% of participants have a third-degree or closer
>     relative in the data, so this throws away a lot of data.
> 2.  **The genetic background is left in the noise**: For a polygenic
>     trait, the many causal SNPs elsewhere in the genome end up in
>     $\epsilon$, so the residual variance is larger than it needs to be
>     and test statistics are smaller.
> 3.  **Rare binary traits**: With far fewer cases than controls (e.g.,
>     1:50), standard logistic regression can give false positives for
>     low-frequency SNPs.
>
> ### Linear Mixed Models (LMMs)
>
> An LMM adds a random **polygenic effect** $g$ to the model:
>
> $$y = X_j \beta_j + Z \gamma + g + \epsilon, \quad g \sim \mathcal{N}(0, \sigma_g^2 K), \quad \epsilon \sim \mathcal{N}(0, \sigma_e^2 I)$$
>
> Here $K$ is the genetic relatedness matrix and $\sigma_g^2$ the
> genetic variance from `GWAS8`. Equivalently, $g = \sum_{k} X_k b_k$ is
> a regression on all SNPs at once (a **whole-genome regression**),
> where the heritability model from `GWAS8` says how large each $b_k$ is
> expected to be.
>
> Modern methods estimate $g$ for each chromosome $c$ from all **other**
> chromosomes ($\hat{g}^{(-c)}$, the `LOCO` prediction). This has two
> effects: - **More power**: Removing $\hat{g}^{(-c)}$ lowers the
> residual variance to about $(1 - R^2_{\text{LOCO}})\,\text{Var}(y)$,
> where $R^2_{\text{LOCO}}$ is the fraction of phenotypic variance that
> the prediction explains. Test statistics then increase by a factor of
> about $1 / (1 - R^2_{\text{LOCO}})$. The maximum is
> $1/(1 - h^2_{\text{SNP}})$, reached only if the genetic background is
> predicted perfectly. With small samples, $R^2_{\text{LOCO}} \approx 0$
> and there is no gain. - **No proximal contamination**: Leaving out the
> tested chromosome means the prediction cannot absorb the signal of the
> tested SNP (or of its LD partners).
>
> ### Comparing Biobank-Scale Mixed-Model Methods
>
> Classical LMMs (`GCTA`, `GEMMA`) build the full $N \times N$
> relatedness matrix $K$, which needs $\mathcal{O}(N^2)$ memory and
> $\mathcal{O}(N^2 M + N^3)$ time—impossible for $N = 500{,}000$. Modern
> methods instead use the two-step whole-genome regression described
> above:
>
> | Method | Heritability Model ($\alpha$) | Step 1 Prior on SNP Effects ($b_k$) | Binary Traits | Computation |
> |---------------|---------------|---------------|---------------|---------------|
> | **PLINK** (`--linear`/`--logistic`) | None (no polygenic term) | None (one SNP at a time) | Logistic regression (Wald test; Firth in PLINK 2) | Fast, $\mathcal{O}(NM)$, but needs unrelated individuals and has less power |
> | **Classical LMM** (`GCTA`, `GEMMA`) | GCTA ($\alpha = -1$) | One Gaussian for all SNPs: $b_k \sim \mathcal{N}(0, \sigma_g^2/M)$ | Linear model or liability approximation | Slow, $\mathcal{O}(N^2 M + N^3)$ |
> | **BOLT-LMM** | GCTA ($\alpha = -1$) | Mixture of two Gaussians (large and small effects) | Linear model approximation | $\mathcal{O}(M N^{1.5})$ |
> | **REGENIE** | GCTA ($\alpha = -1$) | Ridge regression in blocks of SNPs | Firth regression or SPA | Fast, $\mathcal{O}(NM)$ |
> | **LDAK-KVIK** | LDAK model ($\alpha$ chosen from the data, often $-0.25$) | Elastic net (fits both sparse and highly polygenic traits) | Quasi-logistic regression + SPA | Fast and low memory, $\mathcal{O}(NM)$ |
>
> ### How `LDAK-KVIK` Works (Step 1 and Step 2)
>
> 1.  **Step 1 (`--kvik-step1`) — predict the genetic background**:
>     - Estimates $h^2_{\text{SNP}}$ and the expected heritability of
>       each SNP with **randomized Haseman-Elston regression**, which
>       avoids building the $N \times N$ matrix. It tries several values
>       of $\alpha$ and keeps the best (the log reports
>       e.g. `Best power is -0.2500`).
>     - Fits 10 **elastic net** whole-genome regression models (a
>       mixture of a Laplace and a Gaussian prior) with different
>       settings, keeps the one with the lowest mean squared error on
>       10% held-out individuals, and saves 22 `LOCO` predictions
>       $\hat{g}^{(-c)}$, one per chromosome.
> 2.  **Step 2 (`--kvik-step2`) — test each SNP**:
>     - For each SNP $j$ on chromosome $c$, regresses the phenotype on
>       $X_j$ with $\hat{g}^{(-c)}$ removed (included as an offset). For
>       binary traits it uses a quasi-logistic regression and checks
>       small $p$-values with the **Saddlepoint Approximation (SPA)**.
>
> > **Quick Concept Check (from Lecture 9)**: - *What would happen if
> > Step 1 used all 22 chromosomes, including chromosome 1, when testing
> > a causal SNP on chromosome 1?* The prediction would already contain
> > that SNP’s effect, so removing it would also remove the signal
> > (**proximal contamination**: the singer is silenced together with
> > the choir). This is why we use `LOCO`. - *Why can an LMM keep
> > siblings and cousins in the study?* Because the polygenic term
> > $g \sim \mathcal{N}(0, \sigma_g^2 K)$ models the extra genetic
> > similarity between relatives.

# Two-Step Mixed-Model GWAS with `LDAK-KVIK`

### Preparing Phenotype and Covariates

<img src="Images/bash.png" alt="Bash" width="40"> Switch to the Bash
kernel.

First, let’s link the `Data` folder, create `Results/GWAS9`, and remove
the header lines from the simulated phenotype (`Data/SIM.pheno`, true
$h^2_{\text{SNP}} = 0.5$) and covariates (`Data/SIM.covariate`), because
`LDAK` expects files without headers whose first two columns are `FID`
and `IID`:

In [ ]:
ln -sf ../Data
mkdir -p Results/GWAS9

# Strip headers from SIM.pheno and SIM.covariate for LDAK
awk 'NR>1 {print $1, $2, $3}' Data/SIM.pheno > Results/GWAS9/SIM.pheno
awk 'NR>1 {print $0}' Data/SIM.covariate > Results/GWAS9/SIM.covar

echo "Formatted individuals: $(wc -l < Results/GWAS9/SIM.pheno)"

### Running `LDAK-KVIK` Step 1 and Step 2 on Quantitative Traits

Now we run the two steps of `LDAK-KVIK` on the simulated cohort
(`Data/SIM.QC`).

Key `LDAK` options used below: - `--kvik-step1 <prefix>`: Estimates the
heritability, fits the elastic net model (adjusting for the covariates
in `--covar`), and saves the 22 `LOCO` predictions to
`<prefix>.step1.loco.prs`. - `--kvik-step2 <prefix>`: Reads the `LOCO`
predictions from Step 1, tests every SNP in `--bfile`, and writes the
results to `<prefix>.step2.assoc` (and `<prefix>.step2.summaries`). -
`--bfile <prefix>`, `--pheno <file>`, `--covar <file>`: The PLINK
genotype files, the phenotype file and the covariate file (as in
`PLINK`). - `--random-seed 42`: Step 1 uses random numbers (e.g., to
pick the 10% of individuals for cross-validation). Fixing the seed makes
your results very close to ours, although small differences in the last
digits can remain.

In [ ]:
# Note: this step takes less than a minute to run.
# Step 1: Fit Whole-Genome Regression and generate LOCO predictions
ldak \
  --kvik-step1 Results/GWAS9/kvik_sim_trait \
  --bfile Data/SIM.QC \
  --pheno Results/GWAS9/SIM.pheno \
  --covar Results/GWAS9/SIM.covar \
  --random-seed 42

# Step 2: Perform single-SNP association testing conditioned on LOCO predictions
ldak \
  --kvik-step2 Results/GWAS9/kvik_sim_trait \
  --bfile Data/SIM.QC \
  --pheno Results/GWAS9/SIM.pheno \
  --covar Results/GWAS9/SIM.covar

head -n 10 Results/GWAS9/kvik_sim_trait.step2.assoc

The columns of `kvik_sim_trait.step2.assoc` are: - `Chromosome`,
`Predictor` (SNP ID), `Basepair`, `A1` (effect allele), `A2` (other
allele) - `Wald_Stat`: the test statistic ($Z$-score) - `Wald_P`: the
two-sided $p$-value - `Effect`, `SE`: the estimated effect of allele
`A1` ($\hat{\beta}_j$) and its standard error - `MAF`, `CallRate`: the
minor allele frequency and the fraction of non-missing genotypes -
`SPA_Status`: whether the Saddlepoint Approximation was used (binary
traits, see below). For binary traits, `Effect` and `SE` are replaced by
`Approx_Log_OR` and `Approx_SE`

In the Step 1 output above, also look for the line `Best power is ...`
(the chosen $\alpha$ and the heritability estimate) and the
`mean squared error` (MSE) of the 10 models. The phenotype is scaled to
variance 1, so an MSE clearly below 1 means the `LOCO` predictions
explain part of the phenotype
($R^2_{\text{LOCO}} \approx 1 - \text{MSE}$).

# Benchmarking Power: `LDAK-KVIK` vs. Standard `PLINK` GWAS

How much power does a mixed model add? As explained above, test
statistics increase by about $1 / (1 - R^2_{\text{LOCO}})$: the more of
the phenotype Step 1 can predict (at most $h^2_{\text{SNP}}$), the
larger the gain. $R^2_{\text{LOCO}}$ is larger when the sample is large,
the heritability is high, and the trait is driven by fewer causal SNPs.

Let’s test this. We use `ldak --make-phenos` to simulate a new trait on
`Data/SIM.QC` with $h^2_{\text{SNP}} = 0.5$ (`--her 0.5`) and 200 causal
SNPs (`--num-causals 200`) under the LDAK model (`--power -0.25`). The
true causal SNPs are saved in `sim_poly.effects`. We then analyze the
same phenotype with **`PLINK` linear regression** and with
**`LDAK-KVIK`**. *(With only $N \approx 480$ individuals, as in
`Data/EUR.QC`, Step 1 cannot predict anything—all 10 models have an MSE
of about 1—and the two methods give the same results.)*

In [ ]:
# Note: this step takes less than a minute to run.
# 1. Simulate a polygenic trait (h2 = 0.5, 200 causal variants) under the LDAK model
ldak \
  --make-phenos Results/GWAS9/sim_poly \
  --bfile Data/SIM.QC \
  --power -0.25 \
  --her 0.5 \
  --num-phenos 1 \
  --num-causals 200 \
  --random-seed 42

# 2. Run standard single-SNP linear regression in PLINK (with covariates)
plink \
  --bfile Data/SIM.QC \
  --pheno Results/GWAS9/sim_poly.pheno \
  --covar Data/SIM.covariate \
  --linear hide-covar \
  --out Results/GWAS9/plink_sim

# 3. Run LDAK-KVIK Step 1 and Step 2 on the same simulated phenotype
ldak \
  --kvik-step1 Results/GWAS9/kvik_sim \
  --bfile Data/SIM.QC \
  --pheno Results/GWAS9/sim_poly.pheno \
  --covar Results/GWAS9/SIM.covar \
  --random-seed 42

ldak \
  --kvik-step2 Results/GWAS9/kvik_sim \
  --bfile Data/SIM.QC \
  --pheno Results/GWAS9/sim_poly.pheno \
  --covar Results/GWAS9/SIM.covar

# Binary Traits and Saddlepoint Approximation (SPA)

For a case-control phenotype (coded `1` = control and `2` = case, or
`0`/`1`), add `--binary YES` to **Step 1** so that `LDAK-KVIK` fits a
(quasi-)logistic model instead of a linear one. Step 2 detects the
binary phenotype automatically. By default, it rechecks every SNP with
$p < 0.05$ using the **Saddlepoint Approximation (SPA)** (change the
threshold with `--spa-threshold`, or turn SPA off with `--spa-test NO`).
The `SPA_Status` column shows for which SNPs SPA was used.

Why do we need SPA? Standard tests assume that the test statistic
follows a normal distribution under the null hypothesis. When cases are
rare (e.g., 500 cases and 50,000 controls) and a SNP is uncommon, the
true null distribution is skewed, so the normal approximation gives too
many small $p$-values, i.e., false positives. SPA uses the whole shape
of the distribution (including its skewness), so the $p$-values stay
accurate.

Let’s run `LDAK-KVIK` on the case-control phenotype in column 6 of
`Data/TOY_TARGET_DATA.fam` (1,000 cases and 1,000 controls), with
`--binary YES` in Step 1. *(This toy phenotype has little polygenic
signal, so Step 1 reports that it will “exclude the polygenic
contribution”, and Step 2 becomes an ordinary logistic regression with
SPA.)*

In [ ]:
# Note: this step can take several minutes to run.
# Extract binary phenotype (FID IID CaseControl) from TOY_TARGET_DATA.fam
awk '{print $1, $2, $6}' Data/TOY_TARGET_DATA.fam > Results/GWAS9/toy_binary.pheno

# Step 1 with --binary YES
ldak \
  --kvik-step1 Results/GWAS9/kvik_binary \
  --bfile Data/TOY_TARGET_DATA \
  --pheno Results/GWAS9/toy_binary.pheno \
  --binary YES \
  --random-seed 42

# Step 2
ldak \
  --kvik-step2 Results/GWAS9/kvik_binary \
  --bfile Data/TOY_TARGET_DATA \
  --pheno Results/GWAS9/toy_binary.pheno

head -n 5 Results/GWAS9/kvik_binary.step2.assoc

### Visualizing and Comparing `LDAK-KVIK` vs. `PLINK` in R

<img src="Images/R.png" alt="R" width="40"> Switch to the R kernel.

In `R`, we draw the `LDAK-KVIK` Manhattan and Q-Q plots, and compare
`LDAK-KVIK` with `PLINK` at the true causal SNPs (`sim_poly.effects`).

In [ ]:
# Setup to avoid long messages and plot on screen
options(warn = -1)
options(jupyter.plot_mimetypes = 'image/png')
options(repr.plot.width = 10, repr.plot.height = 4.5)

suppressMessages(suppressWarnings(library(ggplot2)))
suppressMessages(suppressWarnings(library(qqman)))

# Load PLINK and LDAK-KVIK results for the simulated polygenic trait
plink_res <- read.table("Results/GWAS9/plink_sim.assoc.linear", header = TRUE)
kvik_res  <- read.table("Results/GWAS9/kvik_sim.step2.assoc", header = TRUE)
causals   <- read.table("Results/GWAS9/sim_poly.effects", header = TRUE)

# Merge by SNP identifier
merged <- merge(plink_res[, c("SNP", "CHR", "BP", "STAT", "P")],
                kvik_res[, c("Predictor", "Wald_Stat", "Wald_P", "Effect", "SE")],
                by.x = "SNP", by.y = "Predictor")

merged$IsCausal <- ifelse(merged$SNP %in% causals$Predictor, "Causal SNP", "Null SNP")
merged$CHISQ_PLINK <- merged$STAT^2
merged$CHISQ_KVIK  <- merged$Wald_Stat^2

cat("Mean Chi-Square at True Causal SNPs (PLINK):    ", round(mean(merged$CHISQ_PLINK[merged$IsCausal == "Causal SNP"], na.rm = TRUE), 3), "\n")
cat("Mean Chi-Square at True Causal SNPs (LDAK-KVIK):", round(mean(merged$CHISQ_KVIK[merged$IsCausal == "Causal SNP"], na.rm = TRUE), 3), "\n")

In [ ]:
# Manhattan and Q-Q plots for LDAK-KVIK
par(mfrow = c(1, 2))
manhattan(kvik_res,
          chr = "Chromosome", bp = "Basepair", p = "Wald_P", snp = "Predictor",
          highlight = causals$Predictor,
          col = c("steelblue", "tomato"),
          main = "LDAK-KVIK Manhattan Plot")

qq(kvik_res$Wald_P, main = "LDAK-KVIK Q-Q Plot", col = "steelblue", pch = 19)

> **Stop - Read - Solve**
>
> - **Q1. Make a scatter plot of `-log10(P)` from `PLINK` (x-axis)
>   against `-log10(Wald_P)` from `LDAK-KVIK` (y-axis), coloring the
>   points by `IsCausal` (`"Causal SNP"` or `"Null SNP"`), and add the
>   line $y = x$ (`geom_abline`).** Which method gives stronger signals
>   at the causal SNPs?
> - **Q2. Compute the genomic inflation factor
>   $\lambda_{\text{GC}} = \text{median}(\chi^2) / \text{qchisq}(0.5, 1)$
>   for `PLINK` and for `LDAK-KVIK`. Why is it important that Step 1
>   uses `LOCO` predictions instead of a prediction from all 22
>   chromosomes?**
>
> Hint: Use `merged` from the previous cell,
> `ggplot(merged, aes(x = -log10(P), y = -log10(Wald_P), color = IsCausal))`,
> and `median(merged$CHISQ_KVIK, na.rm = TRUE) / qchisq(0.5, 1)`.

In [ ]:
# Write your code here


> **Solution**
>
> In `R`, we plot `-log10(P)` from `PLINK` against `-log10(Wald_P)` from
> `LDAK-KVIK`, and compute $\lambda_{\text{GC}}$ for both methods:
>
> ``` bash
> # Q1. Scatter plot of -log10(p) values: PLINK vs. LDAK-KVIK
> p_comp <- ggplot(merged, aes(x = -log10(P), y = -log10(Wald_P), color = IsCausal, alpha = IsCausal)) +
>   geom_point(size = 1.8) +
>   geom_abline(intercept = 0, slope = 1, linetype = "dashed", color = "black") +
>   scale_color_manual(values = c("Causal SNP" = "tomato", "Null SNP" = "steelblue")) +
>   scale_alpha_manual(values = c("Causal SNP" = 0.9, "Null SNP" = 0.25)) +
>   theme_bw(base_size = 13) +
>   labs(
>     title = "Association Power: Standard PLINK vs. LDAK-KVIK",
>     x = expression(-log[10](italic(p)) ~ "(PLINK Linear Regression)"),
>     y = expression(-log[10](italic(p)) ~ "(LDAK-KVIK Mixed Model)"),
>     color = "Variant Class"
>   ) +
>   guides(alpha = "none")
>
> show(p_comp)
>
> # Q2. Genomic inflation factor (lambda_GC)
> lambda_plink <- median(merged$CHISQ_PLINK, na.rm = TRUE) / qchisq(0.5, df = 1)
> lambda_kvik  <- median(merged$CHISQ_KVIK,  na.rm = TRUE) / qchisq(0.5, df = 1)
>
> cat("Lambda_GC (PLINK):    ", round(lambda_plink, 4), "\n")
> cat("Lambda_GC (LDAK-KVIK):", round(lambda_kvik, 4), "\n")
> ```
>
> - **Q1**: The true causal SNPs (red points) lie above the dashed
>   $y = x$ line: removing the `LOCO` prediction makes their test
>   statistics larger. In our run, the mean $\chi^2$ at the 200 causal
>   SNPs increased from about $13.6$ (`PLINK`) to about $21.3$
>   (`LDAK-KVIK`), about 55% more. This agrees with
>   $1/(1 - R^2_{\text{LOCO}})$: the best Step 1 model had an MSE of
>   about $0.65$, so $R^2_{\text{LOCO}} \approx 0.35$ and
>   $1/0.65 \approx 1.5$.
> - **Q2**: Both $\lambda_{\text{GC}}$ values are close to 1 (about
>   $1.02$ for `PLINK` and $1.04$ for `LDAK-KVIK` in our run), so there
>   is no sign of confounding. `LDAK-KVIK` is slightly higher because it
>   increases the test statistics of all SNPs that tag causal variants,
>   which raises the median $\chi^2$ a little. If Step 1 also used the
>   chromosome of the tested SNP, the prediction $\hat{g}$ would already
>   contain that SNP’s effect (directly, or through SNPs in LD with it).
>   Removing $\hat{g}$ from $y$ would then also remove the signal we
>   want to detect (**proximal contamination**), and power would drop. A
>   `LOCO` prediction only removes the background from the other 21
>   chromosomes and leaves the signal on chromosome $c$ untouched.

------------------------------------------------------------------------

### Conclusion

In this tutorial, we saw how mixed-model GWAS (`LDAK-KVIK`) uses SNP
heritability (`GWAS8`) and `LOCO` predictions to increase power for
quantitative and binary traits. Step 2 still tests one SNP at a time, so
its results (`*.step2.assoc` and `*.step2.summaries`) are still blurred
by local LD: many SNPs near a causal variant look significant. In the
next two sessions, we combine **GWAS summary statistics** with an **LD
reference panel** to undo this blurring within a locus (**fine-mapping,
`GWAS10`**) and across the genome (**advanced polygenic scores,
`GWAS11`**).

> **Other take-home messages**
>
> - **Two steps make mixed models fast**: Instead of working with the
>   full $N \times N$ relatedness matrix ($\mathcal{O}(N^3)$),
>   `LDAK-KVIK`, `REGENIE` and `BOLT-LMM` predict the genetic background
>   in Step 1 (`LOCO`) and test single SNPs in Step 2
>   ($\mathcal{O}(NM)$).
> - **Flexible priors**: `LDAK-KVIK` combines the LDAK heritability
>   model ($\alpha$ chosen from the data) with an elastic net prior, so
>   Step 1 works both for highly polygenic traits and for traits with
>   some large-effect SNPs.
> - **Power gains need sample size**: The gain over `PLINK` is about
>   $1/(1 - R^2_{\text{LOCO}})$. It only appears when Step 1 can predict
>   part of the phenotype, which needs thousands of individuals (as in
>   our $N = 5,000$ cohort), not a few hundred.
> - **Rare binary traits**: Use `--binary YES` in Step 1; Step 2 then
>   uses a quasi-logistic model and checks small $p$-values with SPA
>   (see `SPA_Status`), which prevents false positives at low-frequency
>   SNPs.